In [23]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
%autoreload 2
from pathlib import Path
import pandas as pd
import os
import plotly.express as px
from itertools import product

from darpinstances.results import load_aggregate_stats_in_dir, load_occupancies_in_dir

# path and funcs setup

In [30]:
# run_id = "1-run-23-3"
run_id = "2-run-10-4"

In [ ]:
PATH = Path.cwd()
current_path = PATH
INSTANCE_PATH = PATH.parents[3] / "Instances"
RESULTS_PATH = PATH.parents[3] / "Results"
# RESULTS_PATH = PATH.parents[3] / "final-results" / run_id / "Results"

In [ ]:
PATH = PATH.parents[3]
os.chdir(PATH)
IMG_PATH = PATH / "Ridesharing_DARP_instances/figures"

In [40]:
def calculate_avg_method(value, data):
    # average cost across methods
    avg_df = data.groupby(
    ['method', 'area_short', 'duration_minutes', 'max_delay'],
    as_index=False)[value].mean()
    avg_df['is_missing'] = avg_df[value].isna()
    return avg_df

In [41]:
def set_offset(option, data):
    order = sorted(data[option].unique())
    offsets = {
        opt: i - (len(order) - 1) / 2
        for i, opt in enumerate(order)
    }

    vals_to_axis = {}
    i = 1
    delays = sorted(data['max_delay'].unique(), reverse=True)
    durations = sorted(data['duration_minutes'].unique())
    for max_delay in delays:
        for duration in durations:
            vals_to_axis[(duration, max_delay)] = i
            i += 1
    return offsets, vals_to_axis

## results dataframe setup

In [28]:
areas = ['Porto', 'Sydney', 'DC', 'Manhattan', 'Chicago', 'NYC']

### prepare occupancy dataframe

In [32]:
oc_df = pd.DataFrame()
for area in areas:
    res_in_area = load_occupancies_in_dir(RESULTS_PATH / area)
    if res_in_area is None:
        print(f"No results in {area}")
        continue
    res_in_area['area'] = area
    oc_df = pd.concat([oc_df, res_in_area], ignore_index=True)

10:16:20 [INFO] Loading occupancy stats in /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto
10:16:20 [INFO] Loading json file from: /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/vga_chaining/config.yaml-solution.json
10:16:20 [INFO] Loading json file from: /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/vga_chaining/config.yaml-performance.json
10:16:20 [INFO] Loading experiment config from /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/vga_chaining/config.yaml
10:16:20 [INFO] Loading instance config from /home/dominika/Desktop/deathOFbachelor/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_10_min/capacity_6/config.yaml
10:16:20 [INFO] Loading json file from: /home/dominika/Desktop/dea

10:16:20 [INFO] Loading instance config from /home/dominika/Desktop/deathOFbachelor/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_10_min/capacity_10/config.yaml
10:16:20 [INFO] Loading json file from: /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_10/ih/config.yaml-solution.json
10:16:20 [INFO] Loading json file from: /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_10/ih/config.yaml-performance.json
10:16:20 [INFO] Loading experiment config from /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_10/ih/config.yaml
10:16:20 [INFO] Loading instance config from /home/dominika/Desktop/deathOFbachelor/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_10_min/capacity_10/config.yaml
10:16:20 [INFO] Loading json file 

In [33]:
oc_df['cost_per_request'] = oc_df['cost_minutes'] * 60 / oc_df['req_count']
oc_df['area_short'] = oc_df['area'].map({
    'Porto': 'PT',
    'Sydney': 'SY',
    'DC': 'DC',
    'Manhattan': 'MH',
    'Chicago': 'CH',
    'NYC': 'NY'
})
area_order = {
    'Porto': 0,
    'Sydney': 1,
    'DC': 2,
    'Manhattan': 3,
    'Chicago': 4,
    'NYC': 5
}
oc_df['area_order'] = oc_df['area'].map(area_order)
oc_df.sort_values(by=['area_order', 'duration_minutes', 'max_delay', 'method'], inplace=True)
oc_df['method'] = oc_df['method'].replace('halns', 'alns')


### prepare basic dataframe

In [34]:
df = pd.DataFrame()
for area in areas:
    res_in_area = load_aggregate_stats_in_dir(RESULTS_PATH / area)
    if res_in_area is None:
        continue
    res_in_area['area'] = area
    df = pd.concat([df, res_in_area], ignore_index=True)

10:21:56 [INFO] Loading aggregate stats in /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto
10:21:56 [INFO] Loading json file from: /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/vga_chaining/config.yaml-solution.json
10:21:56 [INFO] Loading json file from: /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/vga_chaining/config.yaml-performance.json
10:21:56 [INFO] Loading experiment config from /home/dominika/Desktop/deathOFbachelor/final-results/2-run-10-4/Results/Porto/start_18-00/duration_05_min/max_delay_10_min/capacity_6/vga_chaining/config.yaml
10:21:56 [INFO] Loading instance config from /home/dominika/Desktop/deathOFbachelor/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_10_min/capacity_6/config.yaml
10:21:56 [INFO] Loading json file from: /home/dominika/Desktop/dea

In [35]:
df['cost_per_request'] = df['cost_minutes'] * 60 / df['req_count']
df['area_short'] = df['area'].map({
    'Porto': 'PT',
    'Sydney': 'SY',
    'DC': 'DC',
    'Manhattan': 'MH',
    'Chicago': 'CH',
    'NYC': 'NY'
})
area_order = {
    'Porto': 0,
    'Sydney': 1,
    'DC': 2,
    'Manhattan': 3,
    'Chicago': 4,
    'NYC': 5
}
df['area_order'] = df['area'].map(area_order)
df.sort_values(by=['area_order', 'duration_minutes', 'max_delay', 'method'], inplace=True)
df['method'] = df['method'].replace('halns', 'alns')

# Demand statistics

## Plotly hour demand for PT and SY

In [177]:
from darpinstances.instance_generation.demand_to_database import load_data_from_csv
from darpinstances.instance_generation.convert_formats import RESOURCE_PATH

In [179]:
demand_areas = ["Porto", "Sydney"]

In [182]:
cities_df = pd.DataFrame()
for city in demand_areas:
    csvfile = RESOURCE_PATH / f'{city}_trips.csv'
    city_df = load_data_from_csv(csvfile)
    city_df['area'] = city
    city_df['timestamp'] = pd.to_datetime(city_df['timestamp'])
    cities_df = pd.concat([cities_df, city_df], ignore_index=True)

In [193]:
cities_df['area_short'] = cities_df['area'].map({'Porto': 'PT', 'Sydney': 'SY'})
cities_df['hour'] = cities_df['timestamp'].dt.hour

In [214]:
# Group by hour and calculate the average trip count
hourly_trip_count = cities_df.groupby(['hour', 'area_short']).apply(
    lambda group: group['timestamp'].dt.date.nunique() if group['area_short'].iloc[0] == 'PT' else 1
).reset_index(name='unique_days')

hourly_trip_count = hourly_trip_count.merge(
    cities_df.groupby(['hour', 'area_short']).size().reset_index(name='total_trip_count'),
    on=['hour', 'area_short']
)
# Calculate the percentage of trip count for each area
total_trip_count_by_area = hourly_trip_count.groupby('area_short')['total_trip_count'].transform('sum')
hourly_trip_count['trip_count_percentage'] = (hourly_trip_count['total_trip_count'] / total_trip_count_by_area) * 100
hourly_trip_count['trip_count'] = hourly_trip_count['total_trip_count'] / hourly_trip_count['unique_days']
hourly_trip_count = hourly_trip_count[['hour', 'area_short', 'trip_count', 'trip_count_percentage']].sort_values(by=['area_short', 'hour'])

/tmp/ipykernel_12588/2091659710.py:2: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [ ]:
# Create a bar plot using Plotly
fig = px.line(
    hourly_trip_count,
    x='hour',
    y='trip_count_percentage',
    facet_col='area_short',
)

fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles
fig.for_each_yaxis(lambda x: x.update(title=''))  # Remove titles
fig.add_annotation(x=.5, y=-0.2, text="Hour of the day", xref="paper", yref="paper", showarrow=False)  # x axis
fig.add_annotation(x=-.08, y=0.1, textangle=-90, text="Average proportion of trip [%]", xref="paper", yref="paper", showarrow=False)  # y axis


fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(font=dict(size=28), height=600, width=1300,)

# fig.write_image(f"{IMG_PATH}/hourly_trip_distribution.pdf")

fig.show()

# Methods performance

## Cost

### average travel time - capacity vs method

In [36]:
df_filtered = df[df['cost_per_request'] > 0][['max_delay', 'method', 'area_short', 'cost_per_request', 'duration_minutes', 'capacity']].drop_duplicates()

In [39]:
# Generate all combinations of method, area_short, duration_minutes, and max_delay
methods = df_filtered['method'].unique()
areas = df_filtered['area_short'].unique()
durations = df_filtered['duration_minutes'].unique()
delays = df_filtered['max_delay'].unique()
capacities = df_filtered['capacity'].unique().astype(int)

all_combinations = pd.DataFrame(
    list(product(methods, areas, durations, delays, capacities)),
    columns=['method', 'area_short', 'duration_minutes', 'max_delay', 'capacity']
)

# Merge with the filtered data to find missing combinations
df_complete = all_combinations.merge(df_filtered, on=['method', 'area_short', 'duration_minutes', 'max_delay', 'capacity'], how='left')

df_complete['is_missing'] = df_complete['cost_per_request'].isna()

In [42]:
choose_fighter = 'method'
avg_cost_df = calculate_avg_method('cost_per_request', df_complete)
offsets, vals_to_axis = set_offset(choose_fighter, avg_cost_df)
choose_df = avg_cost_df

In [ ]:
choose_fighter = 'capacity'
choose_df = df_complete[df_complete['method'] == 'ih'].sort_values(by=['area_short', 'duration_minutes', 'max_delay'])

In [49]:
fig = px.histogram(
    choose_df,
    x='area_short',
    y='cost_per_request',
    color=choose_fighter,
    barmode='group',
    facet_col='duration_minutes',
    facet_row='max_delay',
)

# Add X annotations for missing method/area combos
for _, row in avg_cost_df[avg_cost_df['is_missing']].iterrows():
    axis_ref = vals_to_axis[(row['duration_minutes'], row['max_delay'])]

    xref = f'x{axis_ref}' if axis_ref > 1 else 'x'
    yref = f'y{axis_ref}' if axis_ref > 1 else 'y'

    offset = offsets[row[choose_fighter]] * 0.3  # tweak this for spacing
    fig.add_annotation(
        x=row['area_short'],
        y=0,
        text="X",
        xanchor='center',
        yanchor='bottom',
        showarrow=False,
        font=dict(color='black', size=10),
        xref=xref,
        yref=yref,
        xshift=offset * 40  # pixel offset for visual spacing
    )
    # 0.3, 50

# Shared axes titles
fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.add_annotation(
    x=-0.09, y=0.5, text="Average cost per request [s]", textangle=-90,
    xref="paper", yref="paper", showarrow=False
)
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles
fig.add_annotation(
    x=0.19, y=1.12, text="Instance length [min]", xref="paper", yref="paper", showarrow=False
)  # Add shared title
fig.add_annotation(
    x=1.05, y=0.5, text="Maximum delay [s]", xref="paper", yref="paper", showarrow=False, textangle=90
)  # Add shared facet row title

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=0.95), font=dict(size=25), height=800, width=1400,)

fig.write_image(f"{IMG_PATH}/avg_cost_{choose_fighter}.pdf")
fig.show()

### average trip duration

In [15]:
cost_df = oc_df[['area_short', 'max_delay', 'cost_per_request', 'duration_minutes', 'method', 'capacity']].drop_duplicates()

In [22]:
fig = px.histogram(
    cost_df,
    x='cost_per_request',
    color='area_short',
    marginal='box',
    barmode='overlay',
    )

fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles

# titles
fig.add_annotation(x=-0.08, y=0.4, text="Count", textangle=-90, xref="paper", yref="paper", showarrow=False) # y axis title
fig.add_annotation(x=0.5, y=-0.12, text="Average trip duration [s]", xref="paper", yref="paper", showarrow=False) #x axis title
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=0.64, xanchor="right", x=0.99), legend_title_text='Area', font=dict(size=28), height=800, width=1200)

fig.write_image(f"{IMG_PATH}/avg_duration_box.pdf")
fig.show()

### cost diff - ih vs methods

In [50]:
ih_df = avg_cost_df[avg_cost_df['method'] == 'ih'][['area_short', 'duration_minutes', 'max_delay', 'cost_per_request']]
other_methods_df = avg_cost_df[avg_cost_df['method'] != 'ih'][['method', 'area_short', 'duration_minutes', 'max_delay', 'cost_per_request']]

merged = other_methods_df.merge(
    ih_df,
    on=['area_short', 'duration_minutes', 'max_delay'],
    how='outer',
    suffixes=('_other', '_ih')
)

merged.loc[:, 'cost_diff_percentage'] = ((merged['cost_per_request_ih'] - merged['cost_per_request_other']) / merged['cost_per_request_ih']) * 100

merged.loc[:, 'comparison'] = merged.apply(lambda row: f"ih vs {row['method']}", axis=1)

vs_df = merged[['area_short', 'duration_minutes', 'max_delay', 'cost_diff_percentage', 'comparison']]
vs_df.loc[:, 'is_missing'] = vs_df['cost_diff_percentage'].isna()

/tmp/ipykernel_44003/4245659666.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [51]:
offsets, vals_to_axis = set_offset('comparison', vs_df)

In [53]:
fig = px.histogram(
    vs_df,
    x='area_short',
    y='cost_diff_percentage',
    color='comparison',
    barmode='group',
    facet_col='duration_minutes',
    facet_row='max_delay',
)

for _, row in vs_df[vs_df['is_missing']].iterrows():
    axis_ref = vals_to_axis[(row['duration_minutes'], row['max_delay'])]
    
    xref = f'x{axis_ref}' if axis_ref > 1 else 'x'
    yref = f'y{axis_ref}' if axis_ref > 1 else 'y'

    offset = offsets[row['comparison']] * 0.3  # for spacing
    fig.add_annotation(
        x=row['area_short'],
        y=0,
        text="X",
        xanchor='center',
        yanchor='bottom',
        showarrow=False,
        font=dict(color='black', size=10),
        xref=xref,
        yref=yref,
        xshift=offset * 40  # pixel offset for visual spacing
    )

# Shared axes titles
fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.add_annotation(
    x=-0.06, y=0.5, text="Average cost difference [%]", textangle=-90,
    xref="paper", yref="paper", showarrow=False
)
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles
fig.add_annotation(
    x=0.2, y=1.1, text="Instance length [min]", xref="paper", yref="paper", showarrow=False
)  # Add shared title
fig.add_annotation(
    x=1.05, y=0.5, text="Maximum delay [s]", xref="paper", yref="paper", showarrow=False, textangle=90
)  # Add shared facet row title

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=0.95), font=dict(size=20), height=800, width=1300,)

fig.write_image(f"{IMG_PATH}/avg_cost_diff_methods.pdf")
fig.show()


## Delay

In [18]:
delay_area = oc_df[['area_short', 'max_delay', 'avg_delay', 'duration_minutes']]

In [21]:
fig = px.histogram(
    delay_area,
    x='avg_delay',
    color='area_short',
    marginal='box',
    barmode='overlay',
    )

fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles

fig.add_annotation(x=-0.09, y=0.3, text="Count", textangle=-90, xref="paper", yref="paper", showarrow=False) # y axis title
fig.add_annotation(x=0.5, y=-0.1, text="Average delay [s]", xref="paper", yref="paper", showarrow=False) #x axis title
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=0.63, xanchor="right", x=0.99), legend_title_text='Area', font=dict(size=28), height=800, width=1200)

# fig.write_image(f"{IMG_PATH}/avg_delay_box.pdf")
fig.show()

## Occupancy

### occupancy based on max capacity and vehicle hours

In [54]:
oc_df['occupancy_grouped'] = oc_df['occupancy'].apply(lambda x: '6+' if x >= 6 else str(x))

In [57]:
cap_df = oc_df[['area_short', 'vehicle_hours', 'occupancy_grouped', 'method', 'capacity']].drop_duplicates()
cap_df = cap_df.groupby(['area_short', 'method', 'occupancy_grouped', 'capacity'])['vehicle_hours'].sum().reset_index()
total_vehicle_hours = cap_df.groupby(['area_short', 'method', 'capacity'])['vehicle_hours'].transform('sum')
cap_df['vehicle_hours_percentage'] = (cap_df['vehicle_hours'] / total_vehicle_hours) * 100
cap_df = cap_df.sort_values(by=['occupancy_grouped', 'capacity', 'method'])

In [59]:
fig = px.histogram(
    cap_df,
    x='occupancy_grouped',
    y='vehicle_hours_percentage',
    color='method',
    barmode='group',
    facet_col='area_short',
    facet_row='capacity',
)


# small figures
fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles

# titles
fig.add_annotation(x=1.04, y=0.5, text="Max capacity", xref="paper", yref="paper", showarrow=False, textangle=90) # facet row title
fig.add_annotation(x=-0.06, y=0.5, text="Vehicle hours [%]", textangle=-90, xref="paper", yref="paper", showarrow=False) # y axis title
fig.add_annotation(x=0.5, y=-0.08, text="Occupancy", xref="paper", yref="paper", showarrow=False) # x axis title
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=1.03), font=dict(size=28), height=1000, width=1600,)

fig.update_xaxes(tickmode='linear', tick0=0, dtick=1, tickfont=dict(size=18), showticklabels=True)

fig.write_image(f"{IMG_PATH}/hours_occupancy.pdf")
fig.show()

Porto didn't fill the additional capacities (6, 10) - similarly with DC and Chicago (though they managed to reach 7-8 occupancy)
Manhattan, NYC and Sydney managed to reach occupancy 10

## Speed

In [60]:
df_filtered = df[df['cost_per_request'] > 0][['cost_per_request','max_delay', 'method', 'area_short', 'total_time', 'duration_minutes', 'capacity']].drop_duplicates()
df_filtered['comp_time_min'] = df_filtered['total_time'] / 60

# Generate all combinations of method, area_short, duration_minutes, and max_delay
methods = df_filtered['method'].unique()
areas = df_filtered['area_short'].unique()
durations = df_filtered['duration_minutes'].unique()
delays = df_filtered['max_delay'].unique()
capacities = df_filtered['capacity'].unique().astype(int)

all_combinations = pd.DataFrame(
    list(product(methods, areas, durations, delays, capacities)),
    columns=['method', 'area_short', 'duration_minutes', 'max_delay', 'capacity']
)

# Merge with the filtered data to find missing combinations
df_complete = all_combinations.merge(df_filtered, on=['method', 'area_short', 'duration_minutes', 'max_delay', 'capacity'], how='left')

In [61]:
choose_fighter = 'method'
avg_speed_df = calculate_avg_method('comp_time_min', df_complete)
offsets, vals_to_axis = set_offset(choose_fighter, avg_speed_df)

In [63]:
fig = px.bar(
    avg_speed_df,
    x='area_short',
    y='comp_time_min',
    color=choose_fighter,
    barmode='group',
    facet_col='duration_minutes',
    facet_row='max_delay',
    log_y=True,
)
min_y = avg_speed_df.loc[avg_speed_df['comp_time_min'] > 0, 'comp_time_min'].min()
missing_y_val = min_y * 0.5 if min_y else 1e-6 

# Add X annotations for missing method/area combos
for _, row in avg_speed_df[avg_speed_df['is_missing']].iterrows():
    axis_ref = vals_to_axis[(row['duration_minutes'], row['max_delay'])]

    xref = f'x{axis_ref}' if axis_ref > 1 else 'x'
    yref = f'y{axis_ref}' if axis_ref > 1 else 'y'

    offset = offsets[row[choose_fighter]] * 0.3  # for spacing
    fig.add_annotation(
        x=row['area_short'],
        y=-5.5, # for log scale
        text="X",
        xanchor='center',
        yanchor='bottom',
        showarrow=False,
        font=dict(color='black', size=10),
        xref=xref,
        yref=yref,
        xshift=offset * 30  # pixel offset for visual spacing
    )
    # 0.3, 50

# Shared axes titles
fig.for_each_yaxis(lambda y: y.update(title=''))  # Remove titles
fig.add_annotation(
    x=-0.09, y=0.5, text="Average speed [min]", textangle=-90,
    xref="paper", yref="paper", showarrow=False
)
fig.for_each_xaxis(lambda x: x.update(title=''))  # Remove titles
fig.add_annotation(
    x=0.25, y=1.12, text="Instance length [min]", xref="paper", yref="paper", showarrow=False
)  # Add shared title
fig.add_annotation(
    x=1.05, y=0.5, text="Maximum delay [s]", xref="paper", yref="paper", showarrow=False, textangle=90
)  # Add shared facet row title

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=1.05), font=dict(size=25), height=800, width=1300)

fig.write_image(f"{IMG_PATH}/avg_speed_{choose_fighter}.pdf")
fig.show()